In [29]:
import json
from tqdm import tqdm
import random


def flatten_table(table):
    return "\n".join(
        " | ".join(str(cell).strip() for cell in row)
        for row in table
    )


def build_dataset(samples):
    new_data = []
    id_sample = 0
    for sample in tqdm(samples):

        annotation = sample["annotation"]

        # turn hiện tại
        turn_ind = annotation["turn_ind"]

        questions = annotation["dialogue_break"]
        answers = annotation["exe_ans_list"]

        pre_text = " ".join(sample["pre_text"])
        post_text = " ".join(sample["post_text"])
        table_text = flatten_table(sample["table"])

        context_parts = [
            pre_text,
            table_text,
            post_text,
        ]

        # lịch sử QA trước turn hiện tại
        for i in range(turn_ind):
            context_parts.append(
                f"{questions[i]} {answers[i]}"
            )

        # câu hỏi hiện tại
        context_parts.append(
            f"{questions[turn_ind]}"
        )
        new_data.append({
            "input": "\n\n".join(context_parts),
            "label": str(answers[turn_ind])
        })

    return new_data


# Load data gốc
with open("dev_ori.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Build dataset mới
new_data = build_dataset(data)
random.seed(42)      
random.shuffle(new_data)

for idx, item in enumerate(new_data):
    item["id"] = idx

# Save
with open("new_data.json", "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(new_data)} samples to new_data.json")

100%|██████████| 1490/1490 [00:00<00:00, 59635.03it/s]

Saved 1490 samples to new_data.json
